
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Exploring the Knowledge Assistant
## Overview
In this demo, we'll explore the workspace-level Knowledge Assistant that was created during the workspace setup. You'll inspect its configuration in the Agents UI, query it through the Playground, and call its serving endpoint programmatically using the OpenAI client with MLflow tracing.

## Learning Objectives
By the end of this demo, you will be able to:
1. Explore a Knowledge Assistant's configuration in the Agents UI
2. Query a Knowledge Assistant through the AI Playground
3. Call a Knowledge Assistant's serving endpoint programmatically using the OpenAI client
4. Inspect MLflow traces to understand the input/output flow

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Beta Features</strong>
<p style="margin: 8px 0 0 0; color: #333;">This notebook uses Databricks Beta Features. While this notebook has been thoroughly tested, it's worth noting that full functionality is not guaranteed.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup
Run the following cell to set up your environment. This will configure your user catalog and schema.

In [0]:
%run ./Includes/Classroom-Setup-02

## B. Exploring the Knowledge Assistant in the UI

The workspace has a pre-built Knowledge Assistant called **Airbnb Knowledge Assistant** that answers questions about San Francisco Airbnb listings.

1. In the left sidebar, click **Agents**
2. Click on **Airbnb Knowledge Assistant**
3. Take a moment to explore the configuration
    - In the **Build** tab you will see **Sources** and **Settings** on the right-hand side. Your data sources will be found under **Sources** (this KA has a single source called **Airbnb Listings Vector Index) while **Settings** contains your KA's **Instructions** and the **Description** of the agent.
    - The **Examples** tab is a place where you can add labaled questions and guidelines to improve the quality of responses with Agent Learning on Human Feedback (ALHF). This falls outside the scope of this course, but you can read more [here](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant#step-3-improve-quality).


### B1. Query in the Playground

1. In the left sidebar, click **Playground**
2. In the model dropdown at the top, look for the Knowledge Assistant endpoint (it will appear as an Agent endpoint under **Agent Brick**) or you can search for **Airbnb Knowledge Assistant**
3. Click **Use endpoint**
4. Try asking a question like:

<div class="code-block-light" data-language="text">
What are some Airbnb listings near Golden Gate Park?
</div>

<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>

Notice how the response includes **citations**: references back to the source documents. This is a key feature of Knowledge Assistants: every answer is grounded in your data.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">AI Search Error</strong>
<p style="margin: 8px 0 0 0; color: #333;">If the underlying AI search endpoint is not yet ready, you will recieve an error like <code>[ERROR] AI search failed for index 'Airbnb Listings Vector Index': AI search index 'dbacademy.ka_all.airbnb_listings_index' is still provisioning: Delta sync index creation is pending endpoint provisioning. Please wait for provisioning to complete before querying the index.</code> In this scenario, you can navigate to <strong>Compute</strong> on the left and select <strong>AI Search</strong>, where you can monitor the status. 
</div>
</div>
</div>


### B2. Transparent Reasoning
When using the KA in **Playground**, you can toggle **Thoughts** to view the steps in reasoning the agent went through to arrive at its conclusion. It is also best practice to always view the citations for validation. 
1. Hover over the response to see the following;
    - How many seconds it took to generate the first token
    - **View Trace**
    - **Regenerate Response**
    - **Copy Response**
    - **View Sources**
2. Click on **View Trace** 
    - This will display your query, the docs called, and the output from the agent
3. Click on **View Sources**
    - This will show where the references came from (in our case it's a single PDF) as well as the option to view the PDF. 
4. Click on **View PDF** to see the original PDF

## C. Querying the Knowledge Assistant from a Notebook

You can also interact with a Knowledge Assistant programmatically using its model serving endpoint. This is useful for integrating the assistant into applications, pipelines, or automated workflows.


### C1. List Available Knowledge Assistants

Let's first see what Knowledge Assistants exist in this workspace.

The code below uses the **Databricks SDK** to connect to the workspace and retrieve all available Knowledge Assistants:
1. It creates a `WorkspaceClient` instance, which automatically authenticates using the current notebook's credentials.
2. It calls `w.knowledge_assistants.list_knowledge_assistants()` to retrieve all Knowledge Assistants you have access to.
3. It iterates through the results, printing each assistant's **display name** (human-readable) and **ID** (the programmatic identifier used when calling the endpoint).

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# List all knowledge assistants
kas = w.knowledge_assistants.list_knowledge_assistants()
for ka in kas:
    print(f"  Name: {ka.display_name}")
    print(f"  ID:   {ka.name}")
    print()


### C2. Query via the Model Serving Endpoint

Each Knowledge Assistant is served through a model serving endpoint. We can query it using the standard OpenAI-compatible chat completions API.

The code below sets up an **OpenAI client** configured to talk to Databricks Model Serving:
1. It enables **MLflow OpenAI autologging** ([AWS](https://docs.databricks.com/aws/en/mlflow/databricks-autologging) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/mlflow/databricks-autologging) | [GCP](https://docs.databricks.com/gcp/en/mlflow/databricks-autologging)) (`mlflow.openai.autolog()`), which automatically traces all calls made through the OpenAI client, which useful for debugging and observability.
2. It retrieves the current notebook's **API token** from the Databricks context, which authenticates requests to the serving endpoint.
3. It reads the **workspace host URL** from the existing `WorkspaceClient` config.
4. It creates an `OpenAI` client with the token as the API key and `base_url` pointing to `{host}/serving-endpoints`, which routes requests to Databricks Model Serving instead of the public OpenAI API.

In [0]:
import os
from openai import OpenAI
import mlflow

mlflow.openai.autolog()

# The workspace client provides the token and host automatically
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = w.config.host

client = OpenAI(
    api_key=token,
    base_url=f"{host}/serving-endpoints",
)

Let's query the Knowledge Assistant using the OpenAI client we just configured:
1. Retrieve the **serving endpoint name** for the Knowledge Assistant using the Databricks SDK.
2. Call `client.responses.create()` with the endpoint name as the model and our question as input.
3. Because MLflow autologging is enabled, this call is automatically traced. You'll see the trace appear in the cell output.

In [0]:
# Get the serving endpoint name from the Knowledge Assistant
kas = w.knowledge_assistants.list_knowledge_assistants()
for ka in kas:
    print(f"Name: {ka.display_name}, ID: {ka.name}")
    endpoint_name = ka.endpoint_name
    print(f"Endpoint name: {endpoint_name}")

response = client.responses.create(
    model=endpoint_name,
    input=[
        {
            "role": "user",
            "content": "What are some Airbnb listings near Golden Gate Park?"
        }
    ]
)

### C3. Understanding the MLflow Trace

After running the cell above, you should see an **MLflow Trace** displayed in the output. Click on the **Summary** tab to inspect the trace:

- **Input**: Shows the model name (the KA's serving endpoint) and the user's query
- **Output**: Shows the assistant's full response, including the retrieved context and generated answer

This trace is automatically captured by `mlflow.openai.autolog()` and provides end-to-end observability into every call made through the OpenAI client.

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="./Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Let's ask Genie Code to find the endpoint name for the shared Knowledge Assistant. Click on the genie icon <img src="./Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">Use the Databricks SDK to determine the endpoint for the Knowledge Assistants I have access to.</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>


### C4. Try Your Own Questions

Experiment with different questions using both the **Playground** and programmatically. The Knowledge Assistant can answer anything that's covered in the Airbnb listings data. Try questions like the following in the cell below:

<div class="code-block-light" data-language="text">
Which listings mention being close to public transit?
</div>

<div class="code-block-light" data-language="text">
Are there any pet-friendly listings?
</div>

<div class="code-block-light" data-language="text">
What neighborhoods have the most listings?
</div>

<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>

In [0]:
# Try your own question here
my_question = "Which listings mention being close to public transit?"

response = client.responses.create(
    model=endpoint_name,
    input=[
        {
            "role": "user",
            "content": my_question
        }
    ]
)

## D. Conclusion
In this demo, you:

1. Explored the Knowledge Assistant's configuration in the **Agents** UI
2. Queried it through the **Playground** and observed cited responses with transparent reasoning
3. Called the serving endpoint programmatically using the **OpenAI client**
4. Inspected the **MLflow trace** to understand the input/output flow

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
